<a href="https://colab.research.google.com/github/Vasilisa-Kozlovskaya/SANS_itmo/blob/Lab_2_main/SANS_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone -b Lab_2_main https://github.com/Vasilisa-Kozlovskaya/SANS_itmo.git

Cloning into 'SANS_itmo'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 77 (delta 26), reused 48 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 50.35 KiB | 4.20 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [2]:
import os
os.chdir('/content/SANS_itmo')

!ls

 configs     requirements.txt	  src	    'САНС_ЛБ1 (1).ipynb'
 README.md   Sans_modules.ipynb   training


In [3]:
#!pip install -r '/content/SANS_itmo/requirements.txt'
# обновляем torch, transformers и torchvision до стабильных совместимых версий
!pip install --upgrade torch torchvision transformers -q

# станавливаем остальные библиотеки из списка зависимостей
!pip install pytorch-lightning warcio trafilatura langdetect ftfy tokenizers datasets omegaconf clearml python-dotenv -q

# перезапускаем сессию Python
#import os
#os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/2

In [4]:
with open(".env", "w") as f:
    f.write('CHECKPOINT_DIR="./checkpoints"\n')
os.makedirs("./checkpoints", exist_ok=True)

In [ ]:
# @title
!git pull origin Lab_2_main

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 348 bytes | 348.00 KiB/s, done.
From https://github.com/Vasilisa-Kozlovskaya/SANS_itmo
 * branch            Lab_2_main -> FETCH_HEAD
   32afe92..01edbac  Lab_2_main -> origin/Lab_2_main
Updating 32afe92..01edbac
Fast-forward
 configs/configs.yaml | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)


In [ ]:
import sys
import os

# Explicitly add the project root to sys.path
project_root = '/content/SANS_itmo'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.datamodule import CommonCrawlDataModule, WikiTextProcessing, PackedDataset
from src.tokenization.tokenizer import run_tokenization_tasks
from torch.utils.data import DataLoader

WARC_URL = "https://data.commoncrawl.org/crawl-data/CC-NEWS/2025/02/CC-NEWS-20250201012811-00559.warc.gz"
dm = CommonCrawlDataModule(WARC_URL)
dm.prepare_data()

bpe_tokenizer = run_tokenization_tasks(dm.final_data)
wiki_processor = WikiTextProcessing(cc_bpe_tokenizer=bpe_tokenizer)
wiki_texts = wiki_processor.process_wikitext()

packed_data = wiki_processor.create_packed_batches(wiki_texts, block_size=512)

# Инициализируем импортированный датасет
train_dataset = PackedDataset(packed_data)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, drop_last=True)

if len(train_dataset) > 0:
    print(f"Успех! Создано батчей: {len(train_loader)}")

# Возьмем один батч для демонстрации структуры
# x_demo, y_demo, seq_demo = next(iter(train_loader))
# print(f"Формат батча (X): {x_demo.shape}, Формат таргетов (Y): {y_demo.shape}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Cleaning and calculating metrics for WikiText...
WikiText processed. Remaining objects: 20883
Starting Packed Batching (block size: 512)...
Created 6253 packed blocks.
Успех! Создано батчей: 781


In [ ]:
import pytorch_lightning as pl
from src.training.lightning_module import GPTLightningModule
from omegaconf import OmegaConf
from dotenv import load_dotenv

# 1. Загружаем переменные окружения из .env (нужно для интерполяции ${oc.env:...})
load_dotenv()

# 2. ЗАГРУЗКА ИЗ YAML ФАЙЛА
cfg = OmegaConf.load("configs/configs.yaml")

# 3. Переопределяем параметры для быстрого демо-запуска в Colab (опционально)
# Оставляем архитектуру прежней, но уменьшаем шаги обучения, чтобы не ждать часами
cfg.training.max_steps = 50
cfg.training.warmup_steps = 5
cfg.training.batch_size = 8

# 4. Передаем параметры в модель (обращение идет через точку: cfg.model.d_model)
model = GPTLightningModule(
    vocab_size=cfg.model.vocab_size,
    d_model=cfg.model.d_model,
    n_heads=cfg.model.n_heads,
    d_ff=cfg.model.d_ff,
    n_layers=cfg.model.n_layers,
    lr=cfg.training.lr,
    weight_decay=cfg.training.weight_decay,
    warmup_steps=cfg.training.warmup_steps,
    max_steps=cfg.training.max_steps,
    dropout=cfg.model.dropout
)

# 5. Настраиваем Trainer, используя параметры из YAML
trainer = pl.Trainer(
    max_steps=cfg.training.max_steps,
    gradient_clip_val=cfg.training.gradient_clip_val,
    gradient_clip_algorithm="norm",
    accelerator="auto",
    devices=1,
    log_every_n_steps=cfg.training.log_every_n_steps
)

print(f"Конфигурация успешно загружена! Запуск проекта: {cfg.project.name}")
trainer.fit(model, train_loader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


Конфигурация успешно загружена! Запуск проекта: GPT_Language_Model_LR2


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ GPTLanguageModel │ 30.5 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 30.5 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 30.5 M                                                                                               
Total estimated model params size (MB): 122.082                                                                    
Modules in train mode: 92                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/content/SANS_itmo/src/data/datamodule.py:246: UserWarning: To copy construct from a tensor, it is recommended to 
use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than 
torch.tensor(sourceTensor).
  tokens = torch.tensor(block, dtype=torch.long)

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=50` reached.


In [ ]:
import torch

# Переводим модель в режим инференса и переносите на GPU/CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.model.to(device)
model.eval()

# 1. Задаем стартовый текст
prompt = "The science of machine learning"

# 2. Кодируем текст с помощью обученного BPE токенизатора
encoded = bpe_tokenizer.encode(prompt)
input_ids = torch.tensor([encoded.ids], dtype=torch.long, device=device) # Добавляем размерность батча (1, T)

print(f"Промпт: '{prompt}'")
print(f"Токены промпта: {encoded.ids}")

# 3. Вызываем метод generate из вашей модели GPTLanguageModel (описан в grt_model.txt)
# Генерируем 20 новых токенов
with torch.no_grad():
    generated_ids = model.model.generate(input_ids, max_new_tokens=20, temperature=1.0)

# 4. Декодируем результат обратно в текст
# Преобразуем тензор в список id токенов
generated_list = generated_ids[0].cpu().tolist()
decoded_text = bpe_tokenizer.decode(generated_list)

print("\n--- Результат генерации модели ---")
print(decoded_text)


Промпт: 'The science of machine learning'
Токены промпта: [1462, 1713, 4171, 1426, 81, 1623, 1596, 4272]

--- Результат генерации модели ---
The sc ience of m ach ine learning main over Brit 烈 ift teach total , 她


In [6]:
import os
import sys
import torch
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from torch.utils.data import DataLoader
from omegaconf import OmegaConf
from dotenv import load_dotenv

# Explicitly add the project root to sys.path
project_root = '/content/SANS_itmo'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Импортируем ваши модули
from src.data.datamodule import CommonCrawlDataModule, WikiTextProcessing, PackedDataset
from src.tokenization.tokenizer import run_tokenization_tasks
from src.training.lightning_module import GPTLightningModule
from training.train import GradNormLoggerCallback  # Кастомный колбэк из вашего train.txt

def main():
    # 1. Загрузка окружения и конфигурации
    load_dotenv()
    cfg = OmegaConf.load("configs/configs.yaml")

    # Гарантируем наличие папки для чекпоинтов
    os.makedirs(cfg.project.checkpoint_dir, exist_ok=True)

    print(f"--- Запуск полноценного обучения: {cfg.project.name} ---")

    # 2. Полный цикл подготовки данных
    WARC_URL = "https://data.commoncrawl.org/crawl-data/CC-NEWS/2025/02/CC-NEWS-20250201012811-00559.warc.gz"
    dm = CommonCrawlDataModule(WARC_URL)
    dm.prepare_data()

    # Токенизация иPacked Batching
    bpe_tokenizer = run_tokenization_tasks(dm.final_data)
    wiki_processor = WikiTextProcessing(cc_bpe_tokenizer=bpe_tokenizer)
    wiki_texts = wiki_processor.process_wikitext()

    # Разбиваем данные на блоки (длина из конфига или дефолтная 512)
    packed_data = wiki_processor.create_packed_batches(wiki_texts, block_size=512)

    # Разделим данные на train и val для расчета Перплексии
    split_idx = int(len(packed_data) * 0.9)
    train_blocks = packed_data[:split_idx]
    val_blocks = packed_data[split_idx:]

    # 3. Инициализация даталоадеров через PackedDataset
    train_dataset = PackedDataset(train_blocks)
    val_dataset = PackedDataset(val_blocks)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.training.batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.training.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"Размер обучающей выборки (батчей): {len(train_loader)}")
    print(f"Размер валидационной выборки (батчей): {len(val_loader)}")

    # 4. Инициализация Lightning-модели с параметрами из YAML
    model = GPTLightningModule(
        vocab_size=cfg.model.vocab_size,
        d_model=cfg.model.d_model,
        n_heads=cfg.model.n_heads,
        d_ff=cfg.model.d_ff,
        n_layers=cfg.model.n_layers,
        lr=cfg.training.lr,
        weight_decay=cfg.training.weight_decay,
        warmup_steps=cfg.training.warmup_steps,
        max_steps=cfg.training.max_steps,
        dropout=cfg.model.dropout
    )

    # 5. Настройка колбэков (Мониторинг LR, Чекпоинты, Нормы градиентов)
    tb_logger = TensorBoardLogger(save_dir="tb_logs", name=cfg.project.name)

    checkpoint_callback = ModelCheckpoint(
        dirpath=cfg.project.checkpoint_dir,
        filename="best_gpt_model-{epoch:02d}-{val_loss:.2f}",
        monitor="val_loss",  # Если в модуле логируется val_loss
        mode="min",
        save_top_k=1,
        save_last=True      # Сохраняет last.ckpt для возобновления при сбое
    )

    lr_monitor = LearningRateMonitor(logging_interval="step")
    grad_logger = GradNormLoggerCallback() # Будет считать локальные L2 нормы слоев

    # 6. Инициализация PyTorch Lightning Trainer
    trainer = pl.Trainer(
        max_steps=cfg.training.max_steps,
        gradient_clip_val=cfg.training.gradient_clip_val,
        gradient_clip_algorithm="norm",                  # Обрезка по глобальной норме
        logger=tb_logger,
        callbacks=[checkpoint_callback, lr_monitor, grad_logger],
        log_every_n_steps=cfg.training.log_every_n_steps,
        val_check_interval=cfg.training.val_check_interval,
        accelerator="auto",                               # Авто-выбор GPU (T4 в Colab)
        devices=1
    )

    # 7. Проверка возобновления обучения (Задание 2.3)
    last_ckpt_path = os.path.join(cfg.project.checkpoint_dir, "last.ckpt")
    ckpt_path_to_load = last_ckpt_path if os.path.exists(last_ckpt_path) else None

    if ckpt_path_to_load:
        print(f" Найдена сохраненная сессия. Начинаем восстановление с: {ckpt_path_to_load}")

    # 8. Запуск обучения!
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader, ckpt_path=ckpt_path_to_load)
    print(" Обучение успешно завершено!")

if __name__ == "__main__":
    main()


--- Запуск полноценного обучения: GPT_Language_Model_LR2 ---
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Trainable params: 30.5 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 30.5 M                                                                                               
Total estimated model params size (MB): 122.082                                                                    
Modules in train mode: 92                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1000` reached.


 Обучение успешно завершено!


In [8]:
import os
import torch
from omegaconf import OmegaConf
from dotenv import load_dotenv

# Импортируем модули
from src.training.lightning_module import GPTLightningModule
from src.data.datamodule import CommonCrawlDataModule, WikiTextProcessing

def run_inference(prompt_text="The science of machine learning", max_new_tokens=40):
    # 1. Загружаем конфигурацию, чтобы знать параметры архитектуры (d_model, n_layers и т.д.)
    load_dotenv()
    cfg = OmegaConf.load("configs/configs.yaml")

    # 2. Находим путь к лучшему чекпоинту
    # Скрипт обучения сохраняет файл с расширением .ckpt в папку checkpoints/
    checkpoint_dir = cfg.project.checkpoint_dir
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith('.ckpt') and f != 'last.ckpt']

    if not checkpoints:
        # Если кастомных чекпоинтов нет, проверим наличие last.ckpt
        if os.path.exists(os.path.join(checkpoint_dir, 'last.ckpt')):
            checkpoint_path = os.path.join(checkpoint_dir, 'last.ckpt')
        else:
            raise FileNotFoundError(f"В папке {checkpoint_dir} не найдено файлов чекпоинтов (.ckpt)!")
    else:
        # Берем самый свежий/лучший чекпоинт
        checkpoint_path = os.path.join(checkpoint_dir, checkpoints[0])

    print(f"Загрузка весов модели из чекпоинта: {checkpoint_path}")

    # 3. Загружаем модель со всеми весами прямо из чекпоинта
    # PyTorch Lightning сам подставит сохраненные гиперпараметры в __init__
    model = GPTLightningModule.load_from_checkpoint(checkpoint_path)

    # 4. Переносим модель на доступное устройство (GPU/CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # 5. Переводим модель в режим инференса (inference/evaluation mode)
    # Это отключает слои Dropout и фиксирует BatchNorm, если они есть
    model.eval()

    # 6. Подготавливаем токенизатор (он нужен для перевода текста в ID и обратно)
    # Используем BPE токенизатор из пайплайна данных
    WARC_URL = "https://data.commoncrawl.org/crawl-data/CC-NEWS/2025/02/CC-NEWS-20250201012811-00559.warc.gz"
    dm = CommonCrawlDataModule(WARC_URL)
    dm.prepare_data()
    from src.tokenization.tokenizer import run_tokenization_tasks
    bpe_tokenizer = run_tokenization_tasks(dm.final_data)

    # 7. Кодируем промпт в тензор индексов токенов
    encoded = bpe_tokenizer.encode(prompt_text)
    prompt_tokens = encoded.ids

    # Превращаем в тензор и добавляем размерность батча (B=1, T=длина_промпта)
    idx_tensor = torch.tensor([prompt_tokens], dtype=torch.long, device=device)

    print(f"\n--- Результат генерации обученной модели ---")
    print(f"Промпт: '{prompt_text}'")
    print(f"Токены промпта: {prompt_tokens}")

    # 8. Генерация текста (без вычисления градиентов ради экономии памяти)
    with torch.no_grad():
        # Вызываем метод generate вашей внутренней GPT-модели
        # Внутри model (GPTLightningModule) ваша сеть лежит в self.model
        generated_token_ids = model.model.generate(idx_tensor, max_new_tokens=max_new_tokens)

        # Переводим сгенерированные ID токенов обратно в человеческий текст
        # Берем [0], так как батч состоит из одного элемента
        generated_text = bpe_tokenizer.decode(generated_token_ids[0].tolist())

        print(f"Сгенерированный текст: {generated_text}")

# Запуск инференса
run_inference(prompt_text="The science of machine learning", max_new_tokens=50)


Загрузка весов модели из чекпоинта: ./checkpoints/best_gpt_model-epoch=02-val_loss=5.27.ckpt
Using device: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None


Processing 18083 records...


100%|██████████| 18083/18083 [10:44<00:00, 28.05it/s]


Processing and filtering texts...
Dataset average information density: 0.001459

--- Tokenization Tasks ---
Char Vocab Size: 1306
Sample Char Sequence Length: 5284
Word Vocab Size (subset): 7787
Sample Word Sequence Length: 791
BPE Vocab Size: 5000
BPE Encoded Sequence Length: 1358

--- Результат генерации обученной модели ---
Промпт: 'The science of machine learning'
Токены промпта: [1368, 4954, 1332, 81, 1529, 1502, 4178]
Сгенерированный текст: The science of m ach ine learning . part percent OpenAI Wed sh ity favor ves od ug ha art CB from ence sa pite ts also consist h now " , Cong Coun ity Star so pr so can ike num sh e ther sh ace me invol pe Trump AR ily so ity sett
